# DeepAgents 01 · 一个调用拿到「智能体操作系统」（含流式）

整个 `03_deepagents/` 的第一课。目标有两个，一次做完：

1. **`create_deep_agent(...)` 到底给了你什么** —— 不是「又一个 Agent 封装」，
   而是一次调用白送一整套「智能体操作系统」中间件；本课**用内省真的把工具清单打印出来**，
   不靠嘴说。
2. **怎么把它的过程一点点吐出来** —— `agent.stream(..., stream_mode=...)` 的四种模式
   各吐什么、分别用在什么场景。

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| `create_deep_agent` | 「装好全套配件的整车」（发动机是 LangChain 的 `create_agent`） | `create_deep_agent(model=llm, system_prompt=...)` |
| 中间件栈 middleware | 白送的 6 个能力：技能 / 虚拟文件系统 / 子智能体 / 摘要 / 记忆 / 人工审核 | 内省打印出来的工具清单 |
| `StateBackend`（默认后端） | 文件只活在 LangGraph state 里，**不落盘** | `result["files"]` |
| `stream_mode` | 流式的**粒度**：token 级 / 节点级 / 全量状态 / 自定义 | `agent.stream(input, stream_mode=...)` |
| `config={"recursion_limit": 50}` | 深度智能体一步任务要走很多轮，默认 25 步不够 | `agent.invoke(..., config=...)` |

> **本 notebook 由 `Agent/03_deepagents/` 下 4 个脚本合并而成**：
> `01_智能体.py`（课案原版 55 行）、`01_智能体_jxsd.py`（完整版 143 行）、
> `02_流式输出.py`（课案原版 59 行）、`02_流式输出_jxsd.py`（完整版 171 行）。
> 前两个讲「智能体」，后两个讲「流式」—— 原版给最短实现，完整版把同一件事讲透。

**官方文档**
- DeepAgents 总览：<https://docs.langchain.com/oss/python/deepagents/overview>
- DeepAgents 流式：<https://docs.langchain.com/oss/python/deepagents/streaming>
- 底层依赖的 LangGraph 流式：<https://docs.langchain.com/oss/python/langgraph/streaming>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用根目录 `.env` 里配置的大模型 |
| 依赖 | `deepagents` / `langchain` / `langgraph`（本项目 venv 已装） |
| 密钥 | `settings.api_key` / `base_url` / `model_name`（来自根目录 `.env`） |
| 前置服务 | 无 —— 不需要数据库、Docker、MCP、Langfuse |
| 预计耗时 | 约 1~3 分钟（共 7 格会真调模型，且每次任务都是多轮工具调用） |

**本课要花模型额度**：整本 notebook 会真调模型 **7 次左右**（每次任务都要多轮工具调用）。
第一次跑不必慌，掉线/超时就重跑那一格。

> ⚠️ 本课**不往磁盘写任何文件**：默认后端是 `StateBackend`，Agent 的「虚拟文件系统」
> 住在 LangGraph state 里（`result["files"]`），进程一退就没了 —— 这正是下一课
> `02_七种后端.ipynb` 要展开的起点。

## 本节地图

一次 `create_deep_agent` 调用内部发生了什么，以及本课两条主线（智能体 / 流式）的位置：

```mermaid
graph TD
    U["用户一句话"] --> C["create_deep_agent(model=llm, system_prompt=...)"]
    C --> M["deepagent_middleware 中间件栈（白送）"]
    M --> M1["SkillsMiddleware 技能 SKILL.md"]
    M --> M2["FilesystemMiddleware 虚拟文件系统 ls/read/write/edit/glob/grep"]
    M --> M3["SubAgentMiddleware 子智能体 task"]
    M --> M4["SummarizationMiddleware 长上下文自动摘要"]
    M --> M5["MemoryMiddleware 把 /memories/ 注入系统提示词"]
    M --> M6["HumanInTheLoopMiddleware 危险工具调用前挂起"]
    M --> A["一张 LangGraph 图 = create_agent(发动机) + 这套配件"]
    A --> I["agent.invoke(...) 一次性拿最终状态"]
    A --> S["agent.stream(..., stream_mode=...) 一步步吐出来"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 层 | 谁做的 | 本节哪一格能看到 | 对应课案源文件 |
|---|---|---|---|
| 发动机 | `langchain.agents.create_agent`：模型 + 工具 + 中间件组成的基础 Agent 循环 | ——（下一章 `02_langchain/02_智能体与工具.ipynb`） | —— |
| 整车 | `deepagents.create_deep_agent`：先拼中间件栈，最后调 `create_agent` | 第 2 节：内省打印工具清单 | `01_智能体*.py` |
| 一次性执行 | `agent.invoke({"messages": [...]})` | 第 1、2 节 | `01_智能体*.py` |
| 逐步执行 | `agent.stream(input, stream_mode=...)` | 第 3 节四种模式 | `02_流式输出*.py` |

**与上下节的衔接**

- 上一节 `02_langchain/02_智能体与工具.ipynb`：我们用的是 LangChain 的 `create_agent`，
  工具、提示词、中间件全得自己配。
- 本节：换成 `create_deep_agent`，**同一台发动机，配件全给你配好了** ——
  先看清「默认值长什么样」，后面 7 种后端才有的可换。
- 下一节 `03_deepagents/02_七种后端.ipynb`：本节确认默认后端是 `StateBackend`，
  下一节把它依次换成 Store / Filesystem / LocalShell / ContextHub / Sandbox / Composite。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

引导格自己会打印仓库根与临时目录（路径随你把仓库放在哪里而变），输出不固定，故不写预期输出。

> 本课**不会往 `WORKDIR` 里写东西**：Agent 的文件操作走的是 `StateBackend`，
> 内容存在 state 里而不是磁盘上（第 2 节末尾会看到文件清单与磁盘动手的对照）。

下面这格做**前置条件自检**：本课是 🟡 档，硬要求只有两条 ——
**`deepagents` 装好了**、**`.env` 里的模型三件套（key / base_url / model）齐了**。
不满足时打印中文提示，让你知道该补什么，而不是对着一个 `401` 发懵。

In [ ]:
# 前置条件自检：本课需要「已安装 deepagents」+「能调通的模型配置」
from config import settings

try:
    import deepagents  # noqa: F401  —— 只为验证能 import，后面用的是 create_deep_agent
    HAS_DEEPAGENTS = True
    DEEPAGENTS_ERR = ""
except ImportError as exc:
    HAS_DEEPAGENTS = False
    DEEPAGENTS_ERR = str(exc)

HAS_KEY = bool(settings.api_key)
HAS_BASE_URL = bool(settings.base_url)
HAS_MODEL = bool(settings.model_name)
PREREQ_OK = HAS_DEEPAGENTS and HAS_KEY and HAS_BASE_URL and HAS_MODEL

print("deepagents 可导入：", HAS_DEEPAGENTS, DEEPAGENTS_ERR)
print("模型名：", settings.model_name or "（空）")
print("接口地址：", settings.base_url or "（空）")
print("密钥已配置：", HAS_KEY)

if PREREQ_OK:
    print("前置条件满足：本课会真实调用上面的模型，不需要数据库 / Docker / MCP。")
else:
    print("[降级] 缺少依赖或模型配置，请在仓库根 .env 补齐 API_KEY / BASE_URL / MODEL_NAME，"
          "并确认已安装 deepagents，再重新运行本 notebook。")

### 预期输出

```text
deepagents 可导入： True 
模型名： deepseek-flash
接口地址： https://api.deepseek.com
密钥已配置： True
前置条件满足：本课会真实调用上面的模型，不需要数据库 / Docker / MCP。
```

最后一行是**降级分支**：`API_KEY` / `BASE_URL` / `MODEL_NAME` 任一为空、或没装 `deepagents`
时才会出现；出现它就意味着后面的模型调用一定会失败，先去 `.env` 补齐再来。

这一格是**全课唯一确定性输出**，所以它不带任何「易变」声明、`nbtool.py verify` 会真的核对它：
它的内容是「这台机器上的前置条件到底满足没有」。其中 `模型名` / `接口地址` 两行来自根目录
`.env`，本机就是这个值；万一以后换了 `.env` 配置，这两行自然跟着不同，那时重新跑一遍即可。

## 1. 课案原版：最短实现（`01_智能体.py` 55 行）

课案原版只做三件事：**建模型 → 定义一个自己的工具 → 建 Agent 并调一次**。
最短的实现先看一遍，第 2 节再看完整版补了什么。

### 1.1 建模型 + 一个自己的工具

`create_deep_agent` 自己会带一堆**文件与任务类**工具，但你自己的业务函数**必须显式传进去**。
课案这里用 `@tool` 装饰器定义了一个假的联网搜索：

- `@tool` 的作用只有一个：把函数名 / 形参 / 类型注解 / docstring 自动编译成
  **发给模型看的 JSON Schema 描述**（下一章 `04_function_call/` 整章都在讲这件事）；
- 函数体里是**同样的字符串**，生产环境换成 Tavily / SerpAPI 即可。

另外注意课案原文用的是 `ChatOpenAI(model=..., api_key=..., base_url=...)`，
而本项目规范（CONVENTIONS 第 3 节）要求统一走 `init_chat_model`，参数同样来自 `settings`
—— 两者等价，只是后者可以一处改模型、全仓生效。

In [ ]:
# ---------- 1.1 建模型 + 定义自己的工具 ----------
from deepagents import create_deep_agent

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from config import settings

# 课案原文用的是 ChatOpenAI(model=..., api_key=..., base_url=...)；
# 本项目规范（CONVENTIONS 第 3 节）要求统一走 init_chat_model，参数同样来自 settings。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


@tool
def internet_search(query: str) -> str:
    """联网搜索。query：搜索关键词"""
    # 演示用假数据；生产环境可接 Tavily / SerpAPI 等
    return f"搜索「{query}」的结果：LangGraph 是用于构建智能体的图编排框架……"

### 1.2 先备一个小工具：把完整轨迹压成「一行一条」

`print(result)` 打出来是几万字符的完整 state（每条消息的 `response_metadata`、
`usage_metadata`、`tool_calls` 全在里面）—— **信息全，但密度太低**。

所以下面先定义一个 20 行的压缩版：每条消息只打印「谁说的 + 调了哪些工具 + 说了什么开头」。
它是本课自带的调试小工具，第 1、2 节都会用它。

> 注意它是**新增的辅助函数**，不属于课案源文件 —— 课案原文只 `print(result)`。

In [ ]:
# ---------- 1.2 把完整轨迹压缩成「一行一条」 ----------
def print_trace(result: dict) -> None:
    """把 result["messages"] 里每条消息压成一行，便于一眼看出 Agent 干了什么。

    两个判断的依据：
      · tool_calls 非空 → 这条 AI 消息是「模型决定调工具」，把工具名列出来；
      · 否则打印 content 的开头 —— ToolMessage 是工具返回值，AIMessage 是正文。
    """
    for message in result["messages"]:
        kind = type(message).__name__
        calls = [call.get("name") for call in (getattr(message, "tool_calls", None) or [])]
        if calls:
            print(f"  {kind:<14} → 调用工具 {', '.join(calls)}")
        else:
            head = str(getattr(message, "content", "")).replace("\n", " ")[:60]
            print(f"  {kind:<14} {head}")

### 1.3 创建 Agent 并调用：只有 5 行

课案原文到这里一共 5 行。三个细节值得停一下：

1. **没传 `tools` 的「文件工具」** —— `ls` / `read_file` / `write_file` 这些
   `create_deep_agent` 自己会挂上（第 2 节打印出来看）；
2. **`invoke` 的入参格式是 `{"messages": [...]}`**，和 LangGraph 一致；
   课案的 `{"role": "user", "content": ...}` 与元组简写 `("user", ...)` 等价；
3. **`recursion_limit` 必须放宽**：深度智能体一步任务会走很多轮
   （写文件 → 读文件 → 改文件 → 回答），默认 25 步容易触发 `GraphRecursionError`。

⚠️ 下面这格会 `print(result)` 把**完整状态**打出来 —— `result["messages"]` 里
Human / AI / Tool 三种消息全在，所以输出很长。这是课案原文的行为，故意保留：
**看一次完整轨迹**，比只看最后一句有用得多。

In [ ]:
# ---------- 1.3 创建 Agent 并调用（课案原版的 5 行） ----------
agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    system_prompt="你是一个研究助手。先搜索资料，把笔记写入文件再总结。",
)

result = agent.invoke(
    {"messages": [("user", "调研一下 LangGraph 是什么，写成调研笔记")]},
    config={"recursion_limit": 50},  # 深度智能体步数多，放宽步数上限
)
print(result)
print("AI：", result["messages"][-1].content)

# 上面那坨几万字符的 state，用压缩版再看一遍 —— 一眼就能数清「走了几步、调了什么」
print()
print("---------- 同一份 state 的压缩版轨迹 ----------")
print_trace(result)

### 预期输出

> ⚠️ **本段输出完全由模型决定，不要逐字比对。** Agent 每次跑做的事都不一样：它可能直接回答、
> 也可能先 `ls` / `glob` 找一圈再写文件；工具调用轮数、文件路径与内容、消息条数每次运行都不同。
> **稳定的是「事件序列的形状」**—— `Human → AI(带 tool_calls) → ToolMessage → … → AI(最终回答)`
> 这条链，以及 `print(result)` 打出来的一定是**完整 state dict**。别逐字比对正文措辞。

第一行 `print(result)` 的完整输出**约 2 万字符**（里面塞满了 `response_metadata` /
`usage_metadata` / `tool_calls`），这里只留开头一段，`...` 表示我为了放得下而省略的部分：

```text
{'messages': [HumanMessage(content='调研一下 LangGraph 是什么，写成调研笔记', additional_kwargs={}, response_metadata={}),
              AIMessage(content='我先看看文件系统环境，同时开始搜索资料。',
                        response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 2838, ...},
                                           'model_name': 'deepseek-flash', 'finish_reason': 'tool_calls', ...},
                        tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'call_00_USooZecfdgYKSEqhKNTn6530', 'type': 'tool_call'},
                                    {'name': 'internet_search', 'args': {'query': 'LangGraph 是什么 官方文档'}, ...},
                                    {'name': 'internet_search', 'args': {'query': 'LangGraph 核心概念 StateGraph 节点 边'}, ...}],
                        usage_metadata={'input_tokens': 2838, 'output_tokens': 173, 'total_tokens': 3011, ...}),
              ToolMessage(content='No files found', name='ls', ...),
              ToolMessage(content='搜索「LangGraph 是什么 官方文档」的结果：LangGraph 是用于构建智能体的图编排框架……', name='internet_search', ...),
              ...   ← 中间还有好几轮「AIMessage(带 tool_calls) + ToolMessage」往返
              AIMessage(content='笔记已写入 `/LangGraph调研笔记.md`。下面是本次调研总结。…')]}
AI： 笔记已写入 `/LangGraph调研笔记.md`。下面是本次调研总结。
```

紧跟其后的压缩版轨迹才是**用来读的**那一份（上面那 2 万字符，被压成了 27 行）：

```text
---------- 同一份 state 的压缩版轨迹 ----------
  HumanMessage   调研一下 LangGraph 是什么，写成调研笔记
  AIMessage      → 调用工具 ls, internet_search, internet_search
  ToolMessage    No files found
  ToolMessage    搜索「LangGraph 是什么 官方文档」的结果：LangGraph 是用于构建智能体的图编排框架……
  ToolMessage    搜索「LangGraph 核心概念 StateGraph 节点 边」的结果：LangGraph 是用于构建智能体的图编排
  AIMessage      → 调用工具 internet_search, internet_search, internet_search, ls
  ToolMessage    搜索「LangGraph 持久化 checkpointer 人机交互 Human-in-the-loop」的结果：Lan
  ToolMessage    搜索「LangGraph vs LangChain 区别 适用场景」的结果：LangGraph 是用于构建智能体的图编排
  ToolMessage    搜索「LangGraph 安装 快速开始 代码示例 python」的结果：LangGraph 是用于构建智能体的图编排框
  ToolMessage    No files found
  AIMessage      → 调用工具 internet_search, internet_search, internet_search, glob
  ToolMessage    搜索「LangGraph state reducer add_messages conditional edges」的结
  ToolMessage    搜索「LangGraph 多智能体 multi-agent supervisor」的结果：LangGraph 是用于构建
  ToolMessage    搜索「LangGraph Platform LangGraph Studio」的结果：LangGraph 是用于构建智能
  ToolMessage    No files found
  AIMessage      → 调用工具 ls, ls, ls, ls
  ToolMessage    No files found
  ToolMessage    No files found
  ToolMessage    No files found
  ToolMessage    No files found
  AIMessage      → 调用工具 internet_search, internet_search, internet_search
  ToolMessage    搜索「What is LangGraph used for」的结果：LangGraph 是用于构建智能体的图编排框架……
  ToolMessage    搜索「LangGraph 教程 入门」的结果：LangGraph 是用于构建智能体的图编排框架……
  ToolMessage    搜索「LangGraph 缺点 局限 对比」的结果：LangGraph 是用于构建智能体的图编排框架……
  AIMessage      → 调用工具 write_file
  ToolMessage    Updated file /LangGraph调研笔记.md
  AIMessage      笔记已写入 `/LangGraph调研笔记.md`。下面是本次调研总结。  ## 调研总结：LangGraph 是什么
```

**四个值得看的点：**

1. `print(result)` 打出来的是一个 dict，`messages` 字段里**挨着排了 Human → AI（带 `tool_calls`）
   → Tool → AI（最终回答）** —— 这就是一次「会调用工具的 Agent 回合」的完整轨迹；
2. 模型的工具调用**不在最终回答里**，而在中间那些 AI 消息的 `tool_calls` 字段里
   （第 3 节的 `stream_mode="updates"` 会把它单独捞出来打印，好读得多）；
3. 压缩版轨迹一眼可见 **Agent 为了一个「调研」任务跑了 4 轮工具调用**（共 27 条消息）
   —— 每多一轮，就多一条 AI 消息 + 若干 ToolMessage；这就是「深度智能体步数多」这句话的实际含义；
4. 最后一行 `AI：` 后面的文字**每次运行都不完全一样**（模型是概率生成的），
   上面只是我这次跑出来的实测值 —— 你跑出来是不同的句子属于正常现象。

## 2. 完整版：`create_deep_agent` 到底白送了哪些工具

完整版（`01_智能体_jxsd.py`）干的事只有一个：**把「白送了什么」从嘴上说变成打印出来看**。

### 2.1 DeepAgents 是什么

`deepagents` 是 LangChain 官方的「深度智能体」框架，设计目标对标 Claude Code 那一类
「自己开终端、自己读写文件、自己拆任务」的编码智能体。它**不是又一个 Agent 封装**，
而是在基础 Agent 之上**预置了一整套「智能体操作系统」中间件**：

| 中间件 | 给的能力 | 落地成哪些工具 |
|---|---|---|
| `SkillsMiddleware` | 技能：按需加载的 `SKILL.md` 操作手册 | 技能装载相关 |
| `FilesystemMiddleware` | 虚拟文件系统 | `ls` / `read_file` / `write_file` / `edit_file` / `glob` / `grep` / `delete` |
| `SubAgentMiddleware` | 子智能体 | `task` |
| `SummarizationMiddleware` | 长上下文管理：历史过长时自动摘要压缩 | 无（钩子型） |
| `MemoryMiddleware` | 记忆：把 `/memories/` 下的文件注入系统提示词 | 无（钩子型） |
| `HumanInTheLoopMiddleware` | 人工审核：危险工具调用前挂起等批准 | 无（钩子型） |

### 2.2 它和 LangChain `create_agent` 的关系

看 `deepagents/graph.py` 里 `create_deep_agent` 的最后一段：

```python
deepagent_middleware = [SkillsMiddleware(...), FilesystemMiddleware(...),
                        SubAgentMiddleware(...), 摘要中间件, ...]   # 先拼中间件栈
...
return create_agent(                      # ← 最终调用的就是 LangChain 的 create_agent
    model,
    system_prompt=final_system_prompt,
    tools=_tools,
    middleware=deepagent_middleware,
    ...
)
```

也就是说：

```text
langchain.agents.create_agent  = 发动机（模型 + 工具 + 中间件组成的基础 Agent 循环）
deepagents.create_deep_agent   = 装好全套配件的整车（预置了上面那套中间件栈）
```

你完全可以自己用 `create_agent` 手动拼出同样的中间件栈，只是 `deepagents` 帮你把默认值配好了。
所以学 `deepagents` 有个捷径：**看不懂的行为，就去翻 `langchain.agents.middleware` 里的对应中间件**。

### 2.3 默认后端是 `StateBackend`：文件不落盘

源码里就一句：`backend = backend if backend is not None else StateBackend()`。
所以默认情况下文件工具操作的是一个**「住在 LangGraph state 里的虚拟文件系统」**，不碰真实磁盘。

⚠️ 两个由此而来的事实，后面会亲眼验证：

- 工具清单里会出现 `execute`，但**非沙箱后端调用它只会返回错误提示**
  （源码注释原文：`For non-sandbox backends, the execute tool will return an error message.`），
  真正能执行命令的是 06 `LocalShellBackend` / 08 `SandboxBackend`；
- 本轮写进虚拟文件系统的文件，只在 `result["files"]` 里，**磁盘上找不到**。

In [ ]:
# ---------- 只传 model 与 system_prompt，其余全用默认值 ----------
agent = create_deep_agent(
    model=llm,
    system_prompt="你是一个全栈工程师，擅长 Python。",
)

### 2.4 先内省工具清单，再派个真正的活

下面这个 `_list_tools` 是**调试手段，不是业务代码**：

- 它走的路径 `agent.nodes["tools"].bound.tools_by_name` 是 **langgraph 的内部结构**，
  版本升级可能变 —— 所以用 `try/except` 层层兜底，拿不到就返回空列表，
  宁可少打印一行，也不能让整个 notebook 崩掉；
- 「工具清单」这种事实**必须靠内省打印**，不能靠记忆写死：装了什么版本、
  挂了哪些钩子工具，跑一次就有答案。

In [ ]:
# ---------- 内省：看看 create_deep_agent 默认挂了哪些工具 ----------
def _list_tools(agent) -> list[str]:
    """内省：把编译后图里 tools 节点挂载的工具名列出来。

    这只是「看一眼白送了什么」的调试手段，业务代码不需要这么写。
    路径 agent.nodes["tools"].bound.tools_by_name 属于 langgraph 内部结构，
    版本升级可能变，所以用 getattr 层层兜底，拿不到就返回空列表。
    """
    try:
        tools_by_name = agent.nodes["tools"].bound.tools_by_name
        return sorted(tools_by_name)
    except Exception:                      # noqa: BLE001 —— 内省失败不该影响主流程
        return []

现在把这台「整车」开起来：**先看一眼它带了什么工具，再派个活让它自己干**。

任务故意选「写个冒泡排序」：模型有虚拟文件系统，很可能**先写文件、再读回来、最后回答**，
于是我们能顺便看到「一个任务走了多少条消息」。

⚠️ 最后那段 `result.get("files")` 是本课第一个「原来如此」：
**`files` 是 state 里的一个字段，不是磁盘上的文件**。

In [ ]:
# ---------- 打印默认工具清单，然后派个活 ----------
print("create_deep_agent 默认挂载的工具：")
for name in _list_tools(agent):
    print(f"  - {name}")
print()

# 注意 invoke 的入参格式：{"messages": [...]}，和 LangGraph 一致。
# 课案写的是 {"role": "user", "content": ...} 的字典形式；
# 元组形式 ("user", "...") 是 LangChain 的简写，两者等价。
result = agent.invoke(
    {"messages": [{"role": "user", "content": "写个冒泡排序"}]},
    # recursion_limit：深度智能体一步任务会走很多轮（写文件→读文件→改文件→回答），
    # 默认 25 步容易触发 GraphRecursionError，这里放宽到 50。
    config={"recursion_limit": 50},
)

# result["messages"] 是**完整的消息轨迹**（Human / AI / Tool 全在里面），
# 课案只取了最后一条 = Agent 的最终回答。
print("=" * 60)
print("最终回答：")
print(result["messages"][-1].content)

# ---------- 2.4 顺带看一眼 Agent 在虚拟文件系统里留了什么 ----------
# default backend 是 StateBackend，文件存在 state 的 files 字段里 —— 不落盘。
files = result.get("files") or {}
print()
print(f"本轮共 {len(result['messages'])} 条消息；StateBackend 里留下 {len(files)} 个文件")
for path in files:
    print(f"  - {path}")

print()
print("---------- 同一份 state 的压缩版轨迹 ----------")
print_trace(result)

### 预期输出

> ⚠️ **本段输出完全由模型决定，不要逐字比对。** 唯一稳定的是**工具清单**（它是
> `create_deep_agent` 按固定中间件栈挂出来的，同一版本必一致）；**「最终回答」那一大段
> 是模型现写的**，措辞、「本轮共 N 条消息 / N 个文件」全都每次运行都不同
> （本课实测出现过「2 条消息 0 个文件」和「6 条消息 1 个文件」两种结果）。

> 这段输出里**自己带了三反引号的代码块**，所以外面这层围栏用四个反引号包起来。

````text
create_deep_agent 默认挂载的工具：
  - delete
  - edit_file
  - execute
  - glob
  - grep
  - ls
  - read_file
  - task
  - write_file

============================================================
最终回答：
```python
def bubble_sort(nums: list[int]) -> list[int]:
    n = len(nums)
    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):        # 后 i 个已经排好，不用再比
            if nums[j] > nums[j + 1]:
                nums[j], nums[j + 1] = nums[j + 1], nums[j]
                swapped = True
        if not swapped:                   # 本轮无交换 → 已有序，提前退出
            break
    return nums
```

**要点说明**

| 项目 | 说明 |
|---|---|
| 时间 | 平均/最坏 O(n²)；最好 O(n)（已有序，靠 `swapped` 提前退出） |
| 空间 | O(1)，原地排序 |
| 稳定性 | 稳定（`>` 才交换，相等元素相对顺序不变） |
| 优化点 | ① `n - 1 - i` 缩小内层范围；② `swapped` 标志提前结束；③ 记录最后交换位置可进一步缩小边界 |

完整可运行版本（含测试）已写到 `/bubble_sort.py`：

```python
data = [5, 2, 9, 1, 5, 6, 3]
bubble_sort(data)
print(data)   # [1, 2, 3, 5, 5, 6, 9]
```

本轮共 6 条消息；StateBackend 里留下 1 个文件
  - /bubble_sort.py

---------- 同一份 state 的压缩版轨迹 ----------
  HumanMessage   写个冒泡排序
  AIMessage      → 调用工具 ls
  ToolMessage    No files found
  AIMessage      → 调用工具 write_file
  ToolMessage    Updated file /bubble_sort.py
  AIMessage      ```python def bubble_sort(nums: list[int]) -> list[int]:
````

**四个值得看的点：**

1. 工具清单里**没有我们手写的 `internet_search`**（这一节的 Agent 压根没传它），
   全是框架白送的；清单内容随 `deepagents` 版本变化，别把条数当契约（本机实测 9 个）；
2. 这份清单就是中间件栈的「落地结果」：`task` 对应**子智能体**，
   `ls` / `read_file` / `write_file` / `edit_file` / `glob` / `grep` / `delete` 对应
   **虚拟文件系统**，`execute` 对应 shell 执行器；
3. **这一轮是 6 条消息、留下 1 个文件**：对着压缩版轨迹数一下 ——
   `Human → AI(ls) → Tool → AI(write_file) → Tool → AI(回答)`，正好 6 条。
   换一个更「动手」的任务（第 3 节那个「写到 `autumn.md` 里再读出来」）轮数还会更多，
   这就是 `recursion_limit` 必须放宽的原因；
4. `StateBackend 里留下 1 个文件`，路径是 `/bubble_sort.py` —— **磁盘上没有这个文件**，
   它只活在 `result["files"]` 里，去 `ls` 是找不到的。

> 提醒：**这两段输出每次运行都不一样**。同一份代码我第一次跑时，模型选择「直接回答」，
> 于是只有 2 条消息、0 个文件、0 次工具调用；第二次跑才走成上面这样。
> 「模型走几步」是它的自主决定，不是代码写死的 —— 这也正是要留 `recursion_limit` 余量的原因。

## 3. 流式输出：`stream_mode` 的四种模式

第 1、2 节都是 `invoke`（**跑完才给你结果**）。用户等 20 秒什么都看不见，体验很差。
流式的意义就是「**过程可见**」：打字机效果 + 行动轨迹可视化。

### 3.1 四种 `stream_mode`（和 LangGraph 完全一致）

DeepAgents 编译出来的就是一张 LangGraph 图，所以 `graph.stream()` 的用法**一模一样**：

| stream_mode | 每次吐出什么 | 粒度 | 典型用途 |
|---|---|---|---|
| `"messages"` | `(消息块, 元数据)` | **token 级** —— 模型吐一个字就出来一个 | 打字机效果 |
| `"updates"` | `{节点名: 该节点的状态增量}` | **节点级** —— 一步一个 | 看 Agent 调了哪个工具 |
| `"values"` | 每一步之后的**完整状态** | 步骤级 | 调试、观察 `todo` / `files` 的变化 |
| `"custom"` | 节点内部用 `get_stream_writer()` 主动写的自定义事件 | 自定义 | 自定义进度条 |

传一个**列表**可以同时订阅多种模式，此时每个事件会从裸数据变成 `(mode, data)` **二元组**，
调用方必须自己按 mode 分发 —— 这是最容易写错的一步（3.6 会亲眼看到踩法）。

### 3.2 课案代码里的一处注释与参数对不上

课案原文（deepAgents → 流式输出）是这样写的：

```python
# updates：每完成一个节点就输出增量
# for chunk in agent.stream(input_data, stream_mode="messages"):
#     print(chunk[0].content, end="")
```

注释写的是 **updates**，参数用的却是 **`"messages"`** —— 注释是从别处复制过来的。
**按参数为准**：

- `"messages"` 是 **token 级**流式，`chunk[0]` 才是真正的消息块对象，
  `chunk[1]` 是元数据（含 `langgraph_node` 字段，告诉你这个 token 是哪个节点吐的）；
- 「每完成一个节点输出增量」对应的其实是 **`stream_mode="updates"`**。

本课把两种都跑一遍，对照着看就清楚了 —— 原版和完整版是同一件事的两种写法，
不是两个知识点。

### 3.3 拿到 token 级流式的前提

**模型侧必须支持流式返回**。大多数 OpenAI 兼容网关默认就开，完整版显式传了
`streaming=True`，写上更保险 —— 少了它，`stream_mode="messages"` 可能只吐一次一大块，
打字机效果就没了。

In [ ]:
# ---------- 建「流式版」模型与写作助手 Agent ----------
# 这里重新建一个 llm：与第 1 节唯一的差别是多了一个 streaming=True
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    streaming=True,          # 显式开启流式，保证 token 级增量能吐出来
)

# 完整版：不传 system_prompt，也不传 tools，用框架默认的「深度智能体」提示词，
# 顺便看看「不写人设」时它默认是什么风格。
agent = create_deep_agent(model=llm)

# 课案原版：则明确指定「写作助手」人设 + 一张空工具表（只用框架自带工具）
agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="你是写作助手，先在文件里列提纲，再输出正文。",
)

input_data = {"messages": [{"role": "user", "content": "如何制作披萨"}]}

流式块**不能直接 `print(chunk.content)`**：不同模型的 `content` 形态不一样，
直接打印轻则是一坨 Python 对象，重则 `TypeError`。下面这个 `_chunk_text` 把三种形态拍平：

| 形态 | 什么时候出现 | 怎么处理 |
|---|---|---|
| `str` | 普通文本模型 | 直接用 |
| `list` | 多模态 / 带推理的模型，元素可能混着裸字符串与 `{"type": "text", "text": ...}` | 逐块取 `text` 再拼 |
| 其它 / `None` | 纯 reasoning 块、数字等 | 兜底转字符串，空值返回空串 |

两个防御性写法值得学：用 `getattr(chunk, "content", "")` 而不是 `chunk.content`
（有些流式块根本没有 `content` 属性）；用 `item.get("text", "")` 而不是 `item["text"]`
（非文本块——比如图片块——没有 `text` 键）。

In [ ]:
# ---------- 把流式块的内容安全地拍成字符串 ----------
def _chunk_text(chunk) -> str:
    """把消息块的内容安全地转成字符串。

    不同模型的 content 可能是 str，也可能是 [{"type": "text", "text": ...}] 这样的块列表，
    甚至夹杂 reasoning 块。统一拍平成字符串再打印，避免 TypeError。
    """
    # 取出 content；用 getattr 兜底是因为有些流式块（如纯 reasoning 块）根本没有 content
    content = getattr(chunk, "content", "")
    # 形态 1：普通文本模型，content 就是字符串，直接用
    if isinstance(content, str):
        return content
    # 形态 2：多模态 / 带推理的模型，content 是「内容块数组」，
    #         必须逐块取出 text 再拼起来，否则 print 出来是一坨 Python 对象
    if isinstance(content, list):
        parts = []
        for item in content:
            # 数组里可能混着裸字符串
            if isinstance(item, str):
                parts.append(item)
            # 也可能混着 {"type": "text", "text": "..."} 字典；
            # 用 .get("text", "") 而不是 item["text"] —— 非文本块（图片等）没有 text 键
            elif isinstance(item, dict):
                parts.append(str(item.get("text", "")))
        return "".join(parts)
    # 形态 3：其他类型（数字/None 等），兜底转字符串；空值返回空串而不是 "None"
    return str(content) if content else ""

### 3.4 模式①：`stream_mode="messages"` —— token 级打字机

每次拿到的是 `(消息块, 元数据)` 二元组，所以课案写 `chunk[0]`。

⚠️ **`stream_mode="messages"` 吐的不只是「最终回答」**：工具节点的输出、
甚至模型决定调用工具时产生的那一小段文本，也会一起流过来。
下面这格如实打印全部 token，不做过滤 —— 目的就是让你看清「流里到底有什么」。

In [ ]:
# ---------- 模式①：token 级流式（课案原样） ----------
print("===== ① stream_mode='messages'：token 级流式 =====")
for chunk in agent.stream(
    input_data,
    stream_mode="messages",
    config={"recursion_limit": 50},
):
    # chunk 是 (消息块, 元数据) 二元组，课案取 chunk[0] 就是这个消息块
    message_chunk, metadata = chunk[0], chunk[1]
    text = _chunk_text(message_chunk)
    if text:
        print(text, end="", flush=True)
print("\n")

### 预期输出

> ⚠️ **本格输出完全由模型决定，不要逐字比对。** 它可能直接写文件、也可能先 `glob` 找一圈；
> 工具调用次数、文件内容、token 分片的边界每次都不一样。**稳定的是「事件序列的形状」**：
> 「先吐一段过程文本 → 夹入工具节点的输出（如 `No files found` / `Updated file …`）→ 再吐正文」
> 这种 token 级分片的形态不会变，但具体分到哪里、正文写什么，每次运行都不同。

这一格的真实输出共 **3391 字符**（模型写了整整一篇「如何制作披萨」），
这里只留开头一段，`……` 表示省略：

```text
===== ① stream_mode='messages'：token 级流式 =====
No files foundNo files foundUpdated file /pizza_outline.md提纲已写入 `/pizza_outline.md`。下面是正文。

---

# 如何制作披萨

先说一句实话：**在家做出好吃的披萨，难点从来不是"放什么料"，而是面团怎么发酵、烤箱够不够烫。** 只要这两件事对了，哪怕只放番茄和奶酪，也比外卖强。

## 0. 先选你的难度档

| 档位 | 需要什么 | 耗时 | 能达到的水平 |
|---|---|---|---|
| 懒人版 | 平底锅/倒扣烤盘 | 30 分钟 | 家庭快餐水准，解馋够用 |
| 标准版 | 家用烤箱 + 烤石/钢板 | 1–3 天（含发酵） | 接近街边店 |
| 进阶版 | 带烤石的披萨炉 | — | 90 秒出炉，饼边起泡带斑 |

绝大多数人从"标准版"起步最划算。下面按标准版写，懒人版会在第 7 节单独给方案。
……
```

这里可以看到 `"messages"` 模式的两个特点：

1. 输出是**连续不断的字符流**（`end=""` + `flush=True` 的效果），这就是打字机；
2. 因为模型有虚拟文件系统，它**先吐一段「我先写个提纲」之类的过程文本、
   再调工具、再写正文** —— 所以流里会夹着工具节点的输出。
   想要「只要最终回答」，就得按 `metadata["langgraph_node"]` 过滤。

### 3.5 模式②：`stream_mode="updates"` —— 节点级行动轨迹

这才是课案注释里说的那个模式。每次拿到一个 dict：

- **key 是节点名**（`model` / `tools` / 各种中间件钩子）；
- **value 是该节点写回 state 的增量**。

下面这格把增量里的三类消息各打印成一行，好读得多：

| 判断依据 | 是什么 | 打印成 |
|---|---|---|
| `message.tool_calls` 非空 | 模型这一轮**决定**调哪些工具、传什么参数 | `→ 调用工具 X 参数={...}` |
| `type(message).__name__ == "ToolMessage"` | **工具返回**（`ToolMessage.name` 说明是哪个工具） | `← 工具 X 返回：...` |
| 其余有 `content` 的 | AI 的正文（决定不再调工具、直接回答时就是它） | `💬 ...` |

⚠️ 中间那个 `if not messages:` 分支不能省：有些中间件钩子节点**只写非消息字段**，
不打印节点名的话，「每完成一个节点就输出增量」会看起来什么都没发生。

这一格的任务故意用了**文件往返**（写到 `autumn.md` 里再读出来），
好让 `write_file` / `read_file` 两次工具调用都出现在轨迹里。

In [ ]:
# ---------- 模式②：节点级增量（课案注释里说的那个 updates） ----------
print("===== ② stream_mode='updates'：节点级增量（看 Agent 的行动轨迹） =====")
for chunk in agent.stream(
    {"messages": [("user", "写一篇 60 字短文介绍秋天，写到 autumn.md 里再读出来")]},
    stream_mode="updates",
    config={"recursion_limit": 50},
):
    # dict 的 key 是节点名（model / tools / 各中间件钩子），
    # value 是这个节点写回 state 的增量。
    for node_name, update in chunk.items():
        messages = update.get("messages", []) if isinstance(update, dict) else []
        if not messages:
            # 有的中间件钩子节点只写非消息字段，这里照样把节点名报出来，
            # 否则「每完成一个节点就输出增量」会看起来什么都没发生。
            print(f"[{node_name}] （非消息增量）")
            continue
        for message in messages:
            # 工具调用：模型这一轮决定调用哪些工具、传什么参数
            for call in getattr(message, "tool_calls", None) or []:
                print(f"[{node_name}] → 调用工具 {call.get('name')} 参数={call.get('args')}")
            # 工具返回：ToolMessage 自带 name 字段，说明是哪个工具返回的
            if type(message).__name__ == "ToolMessage":
                preview = str(message.content)[:80].replace("\n", " ")
                print(f"[{node_name}] ← 工具 {message.name} 返回：{preview}…")
            elif getattr(message, "content", ""):
                # AI 的正文（模型决定不再调工具、直接回答时就是它）
                preview = str(message.content)[:60].replace("\n", " ")
                print(f"[{node_name}] 💬 {preview}…")

### 预期输出

> ⚠️ **本格输出完全由模型决定，不要逐字比对。** 它可能先 `glob '*.md'` 找文件、也可能直接
> `write_file`；工具调用的**条数与顺序每次都不一样**（本课实测出现过「4 次 ls/glob 摸环境」
> 和「一次 write_file 直接开写」两种轨迹），文件路径与内容也是模型现编的。
> **稳定的是「事件序列的形状」**：`[model] → 调用工具 X` 与 `[tools] ← 工具 X 返回` 这两跳
> 一定成对出现、且 `[model]` 与 `[tools]` 交替；节点名还会随 `deepagents` 版本变。

```text
===== ② stream_mode='updates'：节点级增量（看 Agent 的行动轨迹） =====
[PatchToolCallsMiddleware.before_agent] （非消息增量）
[model] → 调用工具 ls 参数={'path': '/'}
[model] → 调用工具 glob 参数={'pattern': '*.md'}
[tools] ← 工具 ls 返回：No files found…
[tools] ← 工具 glob 返回：No files found…
[model] → 调用工具 ls 参数={'path': '/root'}
[model] → 调用工具 ls 参数={'path': '/tmp'}
[tools] ← 工具 ls 返回：No files found…
[tools] ← 工具 ls 返回：No files found…
[model] → 调用工具 glob 参数={'pattern': '**/*'}
[model] → 调用工具 ls 参数={'path': '/home'}
[tools] ← 工具 ls 返回：No files found…
[tools] ← 工具 glob 返回：No files found…
[model] → 调用工具 write_file 参数={'file_path': '/autumn.md', 'content': '# 秋天（60字短文）\n\n## 提纲\n…（提纲 + 正文全文）\n'}
[tools] ← 工具 write_file 返回：Updated file /autumn.md…
[model] → 调用工具 read_file 参数={'file_path': '/autumn.md'}
[tools] ← 工具 read_file 返回： 1  # 秋天（60字短文）  2    3  ## 提纲  4  1. 起笔：秋风与天光——点出季节，写出天高云淡。  5  2. 气息：秋风的味道——从嗅…
[model] 💬 已完成。文件路径：`/autumn.md`（内容如上，先提纲、后正文，已重读确认）。  **正文（60 字，含标点）：*…
```

对着这张轨迹表，`invoke` 里那些消息是怎么来的就一目了然了：
**每一行 `→` / `←` 都对应 `invoke` 里的一条消息**。

三个细节：

1. **模型先「摸环境」再干活** —— 开头连着 4 次 `ls` / `glob` 都是在找一个能写文件的地方，
   这是深度智能体的典型行为，不是我们写的逻辑；
2. **`[PatchToolCallsMiddleware.before_agent] （非消息增量）`** 正好演示了那句
   「有些中间件钩子节点只写非消息字段」—— 如果没写 `if not messages:` 那个分支，
   这一行就不会出现，看起来像「什么都没发生」；
3. 节点名（`model` / `tools` / `PatchToolCallsMiddleware.before_agent`）会随
   `deepagents` 版本变化，**别把名字当契约**。

### 3.6 模式③：一次订阅多种 —— 事件变成 `(mode, data)` 二元组

传列表订阅多模式后，事件**统一变成 `(模式名, 数据)` 二元组**，
所以必须按模式名分发。这正是多模式订阅**最容易写错的地方**：

```python
# ❌ 照着单模式的写法写 chunk[0]，会把字符串 "messages" 当成消息块
for chunk in agent.stream(input, stream_mode=["messages", "updates"]):
    print(chunk[0].content)      # 这里 chunk[0] 其实是 "messages" 或 "updates"

# ✅ 正确：先 unpack 出 mode，再按 mode 取数据
for mode, data in agent.stream(input, stream_mode=["messages", "updates"]):
    if mode == "messages":
        message_chunk = data[0] if isinstance(data, tuple) else data
```

还有一层嵌套要记牢：**`messages` 模式下的 `data` 又是 `(消息块, 元数据)` 二元组**，
所以要 `data[0]`。两层 `data` 一混，症状就是「打印出来全是英文单词 messages / updates」。

In [ ]:
# ---------- 模式③：多模式同时订阅 ----------
print("\n===== ③ stream_mode=['messages','updates']：一次订阅多种 =====")
for mode, data in agent.stream(
    {"messages": [("user", "用一句话说明什么是递归")]},
    stream_mode=["messages", "updates"],
    config={"recursion_limit": 50},
):
    # 传列表订阅多模式后，事件统一变成 (模式名, 数据) 二元组，
    # 所以这里必须按模式名分发 —— 这正是多模式订阅最容易写错的地方：
    # 照着单模式的写法写 chunk[0]，就会把字符串 "messages" 当成消息块。
    if mode == "messages":
        # messages 模式下 data 是 (消息块, 元数据) 二元组；
        # 用 isinstance 判断是防御性写法，兼容某些版本直接给消息块的情况
        message_chunk = data[0] if isinstance(data, tuple) else data
        text = _chunk_text(message_chunk)
        if text:
            # end="" + flush：token 级流式要的就是「不换行、立刻可见」
            print(text, end="", flush=True)
    else:
        # updates 模式这里只报节点名，避免输出太吵
        print(f"\n[步骤] {list(data.keys())}")
print()

### 预期输出

> ⚠️ **本格输出完全由模型决定，不要逐字比对。** 它先查哪些路径、写哪个文件、写什么内容，
> 以及每个 token 分片落在哪里，每次运行都不一样；`[步骤]` 出现的**次数**也随之变化。
> **稳定的是「事件序列的形状」**：`[步骤] [<节点名>]` 行与 token 级正文**交替**出现，
> 且 `messages` 事件一定夹在 `updates` 事件之间 —— 这才是多模式订阅要看的现象。

```text

===== ③ stream_mode=['messages','updates']：一次订阅多种 =====

[步骤] ['PatchToolCallsMiddleware.before_agent']

[步骤] ['model']
No files found
[步骤] ['tools']
No files found
[步骤] ['tools']

[步骤] ['model']
Updated file /recursion_outline.md
[步骤] ['tools']
提纲已写入 `/recursion_outline.md`（核心机制 → 终止条件 → 一句收尾），正文如下：

**递归就是：一件事情的做法里包含"再按同样的做法处理一个更小的情况"，并且存在一个不再重复、直接给出答案的终止条件。**

（若想更形象：递归就像查词典时解释里又出现同一个词，一路查下去，直到遇到一个不用再查也能懂的词。）
[步骤] ['model']
```

阅读方式：**`[步骤] [...]` 行是「节点级」事件，行内穿插的文字是「token 级」事件**
—— 两种粒度在同一个循环里交替出现，这就是多模式订阅的样子。

两个细节：

1. 最开始的空行 + `[步骤] ['PatchToolCallsMiddleware.before_agent']` 出现在第一段正文之前，
   因为 `messages` 模式这一段还没有 token 可吐；
2. 注意 `③` 输出的**节点名是列表**（`['model']`），而 `②` 里是字典的 key（`[model]`）
   —— 因为这里 `data` 是 `{节点名: 增量}` 的**整个 dict**，我们只取了 `.keys()`。

### 3.7 课案原版的写法对照（`02_流式输出.py`）

完整版的 `_chunk_text` 有 20 行防御代码，课案原版只有 5 行，而且**不处理 `reasoning` 块**。
把原版这一段原样留在这里做对照 —— 两种写法的差别就是「要不要对模型的多种返回形态兜底」：

| | 课案原版 | 完整版 |
|---|---|---|
| `content` 是 `list` 时 | `"".join(str(c) for c in content)` —— 把 dict **整个 `str()` 掉** | 逐块取 `text` 再拼 |
| 没有 `content` 属性的块 | 会 `AttributeError` | `getattr(chunk, "content", "")` 兜底 |
| `None` / 空值 | 可能打印出 `"None"` | 返回空串 |

下面把原版文件**从「建模型」开始整段搬过来**（它自己也建了一份 `llm` 与 `agent`）——
之所以整段搬，就是为了让你亲眼看到：**两个源文件在这一段里只差一句行尾注释**
（原版 `streaming=True,` 后面空着，完整版多写了一句「为什么」）。
重绑一次 `llm` / `agent` 不会影响上面的结果，本节用它跑完最后一格。

> 函数体里的代码**与源文件逐字一致**（连那句「赋值了却没人用」的 `input_data`
> 也保留）—— 它是课案原文的现场，不改动才能对照出完整版补了什么。

In [ ]:
# ---------- 课案原版整段（含它自己的 llm 与 agent，逐字保留） ----------
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    streaming=True,
)

agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt="你是写作助手，先在文件里列提纲，再输出正文。",
)


def course_original_stream() -> None:
    """课案原版（`02_流式输出.py`）的多模式订阅写法，与本节的完整版对照着看。"""
    print("===== token 流式 =====")
    input_data = {"messages": [{"role": "user", "content": "如何制作披萨"}]}

    # updates：每完成一个节点就输出增量
    # for chunk in agent.stream(input_data, stream_mode="messages"):
    #     print(chunk[0].content, end="")

    # 多模式流式：每个事件是 (mode, data) 元组；
    # messages 模式的 data 又是 (消息块, 元数据) 二元组
    for mode, data in agent.stream(
        {"messages": [("user", "写一篇 100 字的短文介绍秋天")]},
        stream_mode=["messages", "updates"],
        config={"recursion_limit": 50},
    ):
        if mode == "messages":
            chunk = data[0] if isinstance(data, tuple) else data
            content = getattr(chunk, "content", "")
            if isinstance(content, list):
                content = "".join(str(c) for c in content)
            if content:
                print(content, end="", flush=True)
        else:
            # 打印每个步骤的节点增量，观察 Agent 的行动轨迹
            print(f"\n[步骤] {data}")


course_original_stream()

### 预期输出

> ⚠️ **本格输出完全由模型决定，不要逐字比对。** 它查哪个目录、写哪个文件、正文写什么，
> 以及 `[步骤]` 出现几次，每次运行都不一样；里面还夹着 `lc_run--…` 这种**随机运行 id**
> 与 `created_at` **时间戳**。**稳定的是「结构」**：`[步骤] {…}` 打出来的是**未经压缩的
> 完整增量 dict**（因此比 `②` 那格长得多），且 `[步骤] {'model': …}` / `[步骤] {'tools': …}`
> 交替出现。

这一格的真实输出共 **4194 字符**（就 8 行 `[步骤]` —— 因为原版把整个 dict 都打了出来）。
下面保留结构，`...` 表示我为了放得下而省略的那些 repr 字段：

```text
===== token 流式 =====

[步骤] {'PatchToolCallsMiddleware.before_agent': None}

[步骤] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-flash', ...}, id='lc_run--01a0ae51-bfcc-...', tool_calls=[{'name': 'ls', 'args': {'path': '/'}, 'id': 'call_00_M6WGWCP5D9jdwDow3jo93379', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2786, 'output_tokens': 99, 'total_tokens': 2885, ...})]}}
No files found
[步骤] {'tools': {'messages': [ToolMessage(content='No files found', name='ls', id='cb7c5b0e-...', tool_call_id='call_00_M6WGWCP5D9jdwDow3jo93379')]}}

[步骤] {'model': {'messages': [AIMessage(content='', ... tool_calls=[{'name': 'glob', 'args': {'pattern': '*'}, ...}])]}}
No files found
[步骤] {'tools': {'messages': [ToolMessage(content='No files found', name='glob', ...)]}}
I'll first create an outline file, then write the essay.
[步骤] {'model': {'messages': [AIMessage(content="I'll first create an outline file, then write the essay.", ... tool_calls=[{'name': 'write_file', 'args': {'file_path': '/autumn_outline.md', 'content': '# 《秋天》短文提纲（约100字）\n\n…'}, ...}])]}}
Updated file /autumn_outline.md
[步骤] {'tools': {'files': {'/autumn_outline.md': {'content': '…', 'encoding': 'utf-8', 'created_at': '2026-09-17T07:43:13.244220+00:00', 'modified_at': '…'}}, 'messages': [ToolMessage(content='Updated file /autumn_outline.md', name='write_file', ...)]}}
已按提纲写好了正文（正文 101 字，含标点）。

**秋天**

秋天是四季中最沉静的季节。天空高远湛蓝，云淡风轻。树叶由绿转黄转红，随风飘落，铺成金黄的小路。田野里稻谷低垂，果园中果实累累，处处是丰收的喜悦。暑气退去，凉风送爽。秋天，既有收获的满足，也有淡淡的思念。

提纲文件已保存为 `/autumn_outline.md`，包含四个层次：总起、景物描写、人的感受、收尾点题。
[步骤] {'model': {'messages': [AIMessage(content='已按提纲写好了正文（正文 101 字，含标点）。\n\n**秋天**\n\n…', ...)]}}
```

和 3.6 的完整版比一比，原版有三个「看不太出来但会咬人」的地方：

1. `print(f"\n[步骤] {data}")` 直接把**整个增量 dict** 打出来 —— 一行几 KB，
   里面还夹着完整的消息对象 repr，所以它这 8 行就有 4194 字符，明显更长更乱；
2. `"".join(str(c) for c in content)` 遇到块列表时，会把 `{"type": "text", ...}`
   这个 dict **整个转成字符串**塞进正文，看起来像乱码；
3. `content` 若为 `None`，`if content:` 能挡住；但若块对象**没有 `content` 属性**，
   `getattr` 兜底（完整版有、原版没有）才是唯一救命的那一层。

## 小结

- **`create_deep_agent` = `create_agent` + 预置中间件栈**。它的问题永远可以回到
  `langchain.agents.middleware` 里找答案；
- **默认后端是 `StateBackend`**：虚拟文件系统住在 LangGraph state 里（`result["files"]`），
  不落盘、也不碰真实磁盘；`execute` 工具在非沙箱后端下只会返回错误提示；
- **`recursion_limit` 要放宽**：一个任务十几到几十条消息都正常，默认 25 步容易撞墙；
- **流式四种模式四种粒度**：`messages`（token 级，打字机）/ `updates`（节点级，行动轨迹）/
  `values`（每步完整状态）/ `custom`（自定义事件）；
- **多模式订阅换来的是 `(mode, data)` 二元组**，必须自己分发 —— 而 `messages` 模式下的
  `data` 又是 `(消息块, 元数据)`，是第二个坑；
- **`streaming=True` 是 token 级流式的前提**，靠网关默认值不保险。

下一课 `02_七种后端.ipynb`：本节确认了默认后端是 `StateBackend`，
下一节把它依次换成 Store / Filesystem / LocalShell / ContextHub / Sandbox / Composite ——
让 Agent 的文件真正落到内存、磁盘、命令沙箱里去。

## 常见坑

1. **`GraphRecursionError`** —— 深度智能体一步任务要走很多轮，
   `invoke` / `stream` 都记得带 `config={"recursion_limit": 50}`。
2. **`execute` 工具「存在但没用」** —— 非沙箱后端调用它只会返回错误提示。
   要真执行命令，得换 06 `LocalShellBackend` / 08 `SandboxBackend`。
3. **以为文件写到磁盘上了** —— 默认 `StateBackend` 不落盘。
   去 `ls` 找不到不是 bug，去 `result["files"]` 找。
4. **多模式订阅时把 `mode` 当成消息块** —— 传列表后事件是 `(mode, data)`；
   另外 `messages` 模式的 `data` 还是 `(消息块, 元数据)`，要 `data[0]`。
5. **`print(chunk.content)` 直接打印流式块** —— `content` 可能是 `list` 甚至不存在，
   统一走 `getattr(..., "")` + 逐块取 `text`。
6. **忘了 `streaming=True`** —— 网关默认值各不相同，写死了才不会「流式退化成一次性」。
7. **`_list_tools` 拿不到工具清单** —— 它走的是 langgraph 内部结构，
   版本升级会变。所以它只用来「看一眼」，必须层层兜底，不能进业务逻辑。

## 官方链接

- DeepAgents 总览：<https://docs.langchain.com/oss/python/deepagents/overview>
- DeepAgents 流式：<https://docs.langchain.com/oss/python/deepagents/streaming>
- LangGraph 流式（四种 `stream_mode` 的原始定义）：<https://docs.langchain.com/oss/python/langgraph/streaming>
- LangChain Agents（底层发动机 `create_agent`）：<https://docs.langchain.com/oss/python/langchain/agents>
- LangChain 工具（`@tool` 与 JSON Schema）：<https://docs.langchain.com/oss/python/langchain/tools>